In [3]:
import torch
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris, load_wine, load_breast_cancer, load_digits
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score

from scipy.optimize import linear_sum_assignment
from PhiFCM import PhiFCM
from PhiKmeans import PhiKMeans

# Data
def load_iris_data():
    data = load_iris()
    return data.data, data.target

def load_wine_data():
    data = load_wine()
    return data.data, data.target

def load_cancer_data():
    data = load_breast_cancer()
    return data.data, data.target

datasets = {
    "iris": load_iris_data,
    "wine": load_wine_data,
    "cancer": load_cancer_data,
}

# Align labels
def align_labels(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    D = max(y_pred.max(), y_true.max()) + 1
    cost = np.zeros((D, D))
    for i in range(len(y_true)):
        cost[y_pred[i], y_true[i]] += 1
    row, col = linear_sum_assignment(-cost)
    mapping = {r: c for r, c in zip(row, col)}
    return np.array([mapping[l] for l in y_pred])

datasets = {
    "iris": load_iris,
    "wine": load_wine,
    "cancer": load_breast_cancer,
}

# Alignement
def align_labels(y_true, y_pred):
    D = max(y_pred.max(), y_true.max()) + 1
    cost = np.zeros((D, D))
    for i in range(len(y_true)):
        cost[y_pred[i], y_true[i]] += 1
    row, col = linear_sum_assignment(-cost)
    mapping = {r: c for r, c in zip(row, col)}
    return np.array([mapping[l] for l in y_pred])


In [4]:
# XP settings
device = "cuda" if torch.cuda.is_available() else "cpu"
alphas = np.arange(0.1, 2.0, 0.1)
m_values = [1.1, 1.3, 1.5, 2.0]  # include values close to 1
results = []

#############

for name, loader in datasets.items():
    print(f"\nDataset: {name}")

    data = loader()
    X = StandardScaler().fit_transform(data.data)
    y = data.target
    X = torch.tensor(X, dtype=torch.float32)
    k = len(np.unique(y))
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # SAME initialization for all (alpha, m)
        init_centroids = PhiKMeans.kmeans_pp_init(X_train, k)
        best_score = np.inf
        best_model = None
        best_alpha = None
        best_m = None

        for alpha in alphas:
            for m in m_values:
                model = PhiFCM(k, alpha=alpha, m=m, device=device)
                model.fit(X_train, init_centroids)
                # FS selection 
                fs = model.fukuyama_sugeno(
                    model.phi(X_train, alpha),
                    model.U,
                    model.W
                )

                if fs < best_score:
                    best_score = fs
                    best_model = model
                    best_alpha = alpha
                    best_m = m

        # Test evaluation
        pred = best_model.predict(X_test).cpu().numpy()
        pred = align_labels(y_test, pred)
        precision = precision_score(y_test, pred, average="macro")
        recall = recall_score(y_test, pred, average="macro")
        f1 = f1_score(y_test, pred, average="macro")

        results.append({
            "dataset": name,
            "fold": fold,
            "alpha": best_alpha,
            "m": best_m,
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

        print(
            f"Fold {fold} | alpha={best_alpha:.2f} | m={best_m:.1f} | F1={f1:.4f}"
        )

#############

# Results
df = pd.DataFrame(results)
print("\nMean results per dataset:")
print(df.groupby("dataset")[["precision", "recall", "f1"]].mean())


Dataset: iris
Fold 0 | alpha=0.20 | m=1.5 | F1=0.7273
Fold 1 | alpha=0.20 | m=1.3 | F1=0.7489
Fold 2 | alpha=0.20 | m=1.3 | F1=0.7177

Dataset: wine
Fold 0 | alpha=1.40 | m=1.1 | F1=0.9673
Fold 1 | alpha=0.90 | m=1.1 | F1=0.9690
Fold 2 | alpha=1.20 | m=1.1 | F1=0.9510

Dataset: cancer
Fold 0 | alpha=1.90 | m=2.0 | F1=0.8820
Fold 1 | alpha=1.90 | m=2.0 | F1=0.8841
Fold 2 | alpha=1.90 | m=2.0 | F1=0.8015

Mean results per dataset:
         precision    recall        f1
dataset                               
cancer    0.856201  0.876425  0.855873
iris      0.829997  0.767157  0.731272
wine      0.960954  0.967190  0.962421
